# M3 · 少计算：采样步数与 Amdahl 定律

**核心问题：** 40 步变 8 步，为什么端到端不是整整快 5 倍？用真实 DiT 占比推算 Amdahl 上限，再与 L1 实测比较。Turbo 会改变模型/调度，画质不能自动判为通过。

**学习闭环：** 先预测 → 检查固定证据 → 独立计算 → 批判性解释 → 设计受控实验。

对应理论课：[打开 course/M03_less_computation/LESSON.zh-CN.md](../course/M03_less_computation/LESSON.zh-CN.md)。切换语言：[English](./03_less_computation.ipynb)。

**默认模式：** 纯 CPU、无需联网、只读仓库证据。不会自动启动 GPU。


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root (see notebooks/README).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from lab import notebook_support as ns
import statistics
print("Course repository:", ROOT)
print("Mode: OFFLINE / committed reference evidence; NO GPU work is launched by default.")


## 检查真实数据并独立计算

按照顺序执行；每个比例都要解释分母与适用范围。

In [ ]:
rows = ns.summary_rows()
b, l = rows["B0"], rows["L1"]
observed = b["warm_total_median_s"] / l["warm_total_median_s"]
p = b["warm_ksampler_median_s"] / b["warm_total_median_s"]
upper = 1 / ((1 - p) + p / 5)
print(f"B0={b['warm_total_median_s']:.3f}s, L1={l['warm_total_median_s']:.3f}s")
print(f"Measured speedup={observed:.3f}x; Amdahl estimate at 5x denoiser={upper:.3f}x")
assert 4 < observed < 5
print("These are distinct model/schedule variants; the images are NOT lossless parity.")
plot_values = {"B0": b["warm_total_median_s"], "Turbo L1": l["warm_total_median_s"]}
plot_ylabel, plot_title = "warm seconds", "Reduce sampling work (not a lossless edit)"


In [ ]:
SHOW_PLOT = False  # Optional: requires matplotlib in the notebook kernel
if SHOW_PLOT:
    import matplotlib.pyplot as plt
    labels = list(plot_values)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(labels, [plot_values[label] for label in labels])
    ax.set_ylabel(plot_ylabel)
    ax.set_title(plot_title)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print("Optional plot skipped; set SHOW_PLOT=True to visualize the measured values.")


## 可选 GPU 真机冒烟——必须显式授权

需要 ROCm 环境、模型与 GPU 额度。CI 不执行；结果只存于 `runs/`，不得混入官方证据。

In [ ]:
RUN_GPU = False  # Only switch on after checking platform quota + available models
if RUN_GPU:
    # Also set QIR_NOTEBOOK_GPU_ACK=YES in your own kernel environment.
    # The helper enforces --smoke and writes under runs/, never evidence/.
    ns.gpu_smoke("L1", "notebook-l1", confirmed=True)
else:
    print("GPU NOT_RUN; evidence analysis above is valid without GPU access.")


## 学生思考 / 自检

把公式中去噪部分的理论加速设为 2× 和 10×，分析瓶颈迁移。为什么调度变更可能提高速度却改变图像内容？

**证据规范：** 用自己的语言完成回答。本 Notebook 重算的是已提交记录，不是你的 GPU 实测。如果自愿运行 smoke，结果必须放在 `runs/`，不能宣传为正式全量统计。
